
# MATH500 Reasoning Accuracy Verifier — `HuggingFaceTB/SmolLM-360M`

This notebook builds an end-to-end **evaluation + verification pipeline** for measuring how
well a small language model solves the [MATH500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500)
benchmark (a 500-problem subset of the MATH dataset spanning algebra, geometry, number theory,
calculus, probability, etc.).

**Pipeline overview**

1. Load the MATH500 dataset.
2. Load `HuggingFaceTB/SmolLM-360M` (a 360M-parameter **base**, non-instruction-tuned model).
3. Prompt it with a few-shot chain-of-thought template so a base model has a chance at
   producing a parseable final answer.
4. Generate completions.
5. **Extract** the model's final answer (from `\boxed{...}` or fallback heuristics).
6. **Verify** the extracted answer against the ground truth using a math-aware equivalence
   checker (handles fractions, decimals, algebraic simplification, ordering of tuples, etc.)
7. Aggregate accuracy overall, by difficulty level, and by subject — with plots and an
   error-analysis view of the worst failure cases.

> ⚠️ **Expectation-setting**: `SmolLM-360M` is a *base* model, not instruction-tuned, and is
> tiny by reasoning-benchmark standards. Expect low absolute accuracy (plausibly close to,
> or not far above, the parsing-noise floor). This notebook is written so the *pipeline* is
> correct and reusable — swap in `HuggingFaceTB/SmolLM2-360M-Instruct`,
> `HuggingFaceTB/SmolLM2-1.7B-Instruct`, or any other `transformers` causal LM by changing
> one config variable, and the rest works unchanged. That comparison is set up in the last
> section.


## 1. Environment setup

In [ ]:

# Run once. Safe to re-run.
%pip install -q -U "transformers>=4.44" "datasets>=2.20" accelerate sympy antlr4-python3-runtime==4.11 tqdm matplotlib pandas


In [ ]:

import os, re, json, math, time, random, textwrap, warnings
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

warnings.filterwarnings("ignore")
set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 2. Configuration

Everything you're likely to want to tweak lives here.

In [ ]:

@dataclass
class EvalConfig:
    model_name: str = "HuggingFaceTB/SmolLM-360M"   # swap this for any causal LM
    dataset_name: str = "HuggingFaceH4/MATH-500"     # 500-problem MATH test subset
    dataset_split: str = "test"

    num_samples: Optional[int] = 100   # None -> run the full 500. Start small to sanity check.
    num_few_shot: int = 4              # few-shot CoT exemplars prepended to each prompt
    max_new_tokens: int = 512
    temperature: float = 0.0           # 0.0 -> greedy decoding (deterministic, recommended for eval)
    top_p: float = 1.0
    batch_size: int = 8                # reduce if you hit OOM on CPU/small GPU
    seed: int = 42

    dtype: str = "auto"                # "auto" | "float32" | "float16" | "bfloat16"
    output_dir: str = "./math500_eval_results"

config = EvalConfig()
os.makedirs(config.output_dir, exist_ok=True)
config



## 3. Load MATH500

The canonical HF mirror is `HuggingFaceH4/MATH-500`, with fields `problem`, `solution`,
`answer`, `subject`, `level`, `unique_id`. We fall back to `qwedsacf/competition_math`-style
loading logic if the primary dataset id is unavailable in your environment.


In [ ]:

def load_math500(cfg: EvalConfig):
    try:
        ds = load_dataset(cfg.dataset_name, split=cfg.dataset_split)
    except Exception as e:
        print(f"Primary dataset load failed ({e}); trying alternate id 'HuggingFaceH4/MATH-500' main config...")
        ds = load_dataset("HuggingFaceH4/MATH-500")[cfg.dataset_split]

    # Normalize column names across possible dataset variants
    cols = set(ds.column_names)
    rename_map = {}
    if "problem" not in cols and "question" in cols:
        rename_map["question"] = "problem"
    if "answer" not in cols and "final_answer" in cols:
        rename_map["final_answer"] = "answer"
    if rename_map:
        ds = ds.rename_columns(rename_map)
    return ds

full_dataset = load_math500(config)
print(f"Loaded {len(full_dataset)} problems.")
print("Columns:", full_dataset.column_names)

if config.num_samples is not None:
    eval_dataset = full_dataset.shuffle(seed=config.seed).select(range(min(config.num_samples, len(full_dataset))))
else:
    eval_dataset = full_dataset

print(f"Evaluating on {len(eval_dataset)} problems.")


In [ ]:

# Peek at a couple of examples
for i in range(2):
    ex = eval_dataset[i]
    print("=" * 80)
    print("PROBLEM:", ex["problem"][:400])
    print("-" * 80)
    print("GROUND TRUTH ANSWER:", ex.get("answer"))
    print("LEVEL:", ex.get("level"), "| SUBJECT:", ex.get("subject") or ex.get("type"))


## 4. Load the model

In [ ]:

dtype_map = {
    "auto": "auto",
    "float32": torch.float32,
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
}

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched causal-LM generation

model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=dtype_map[config.dtype],
).to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {config.model_name}: {n_params/1e6:.1f}M parameters on {DEVICE}")



## 5. Prompt template (few-shot chain-of-thought)

`SmolLM-360M` is a **base** model — it was never fine-tuned to follow instructions or to stop
after answering. Few-shot examples that model the exact behavior we want (reason, then emit
`The final answer is \boxed{...}.` and stop) give it the best realistic shot at producing a
parseable answer. We cut generation off at the first exemplar-style delimiter to avoid the
model rambling into a new, unrelated "problem".


In [ ]:

FEW_SHOT_EXEMPLARS = [
    {
        "problem": "What is $1+2+3+\\cdots+10$?",
        "solution": "The sum of the first $n$ positive integers is $\\frac{n(n+1)}{2}$. "
                     "For $n=10$ this is $\\frac{10 \\cdot 11}{2} = 55$. "
                     "The final answer is $\\boxed{55}$.",
    },
    {
        "problem": "Simplify $\\frac{6}{8}$.",
        "solution": "The greatest common divisor of $6$ and $8$ is $2$, so "
                     "$\\frac{6}{8} = \\frac{6/2}{8/2} = \\frac{3}{4}$. "
                     "The final answer is $\\boxed{\\frac{3}{4}}$.",
    },
    {
        "problem": "If $f(x) = 2x + 3$, what is $f(5)$?",
        "solution": "Substituting $x=5$: $f(5) = 2(5) + 3 = 10 + 3 = 13$. "
                     "The final answer is $\\boxed{13}$.",
    },
    {
        "problem": "Solve for $x$: $2x - 4 = 10$.",
        "solution": "Adding $4$ to both sides gives $2x = 14$. Dividing by $2$ gives $x = 7$. "
                     "The final answer is $\\boxed{7}$.",
    },
]

PROMPT_HEADER = (
    "Solve the following math problems step by step. "
    "End every solution with a sentence of the exact form "
    '"The final answer is $\\boxed{ANSWER}$."\n\n'
)

def build_prompt(problem: str, n_shot: int) -> str:
    shots = FEW_SHOT_EXEMPLARS[:n_shot]
    parts = [PROMPT_HEADER]
    for ex in shots:
        parts.append(f"Problem: {ex['problem']}\nSolution: {ex['solution']}\n\n")
    parts.append(f"Problem: {problem}\nSolution:")
    return "".join(parts)

print(build_prompt(eval_dataset[0]["problem"], config.num_few_shot))


## 6. Batched generation

In [ ]:

STOP_STRINGS = ["\nProblem:", "\n\nProblem:"]

def truncate_at_stop(text: str, stops: List[str]) -> str:
    idx = len(text)
    for s in stops:
        pos = text.find(s)
        if pos != -1:
            idx = min(idx, pos)
    return text[:idx]

@torch.no_grad()
def generate_batch(prompts: List[str], cfg: EvalConfig) -> List[str]:
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(DEVICE)
    gen_kwargs = dict(
        max_new_tokens=cfg.max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=cfg.temperature > 0.0,
    )
    if cfg.temperature > 0.0:
        gen_kwargs.update(temperature=cfg.temperature, top_p=cfg.top_p)
    output_ids = model.generate(**inputs, **gen_kwargs)
    input_len = inputs["input_ids"].shape[1]
    completions = tokenizer.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)
    return [truncate_at_stop(c, STOP_STRINGS) for c in completions]



## 7. Answer extraction

MATH-style answers are usually wrapped in `\boxed{...}`. We implement a brace-matching parser
(a regex alone breaks on nested braces like `\boxed{\frac{1}{2}}`), with fallbacks for models
that drop the `\boxed` wrapper.


In [ ]:

def extract_boxed(text: str) -> Optional[str]:
    # Find the *last* \boxed{...} in text, correctly handling nested braces.
    key = "\\boxed{"
    start = text.rfind(key)
    if start == -1:
        return None
    i = start + len(key)
    depth = 1
    buf = []
    while i < len(text) and depth > 0:
        ch = text[i]
        if ch == "{":
            depth += 1
            buf.append(ch)
        elif ch == "}":
            depth -= 1
            if depth > 0:
                buf.append(ch)
        else:
            buf.append(ch)
        i += 1
    if depth != 0:
        return None  # unbalanced braces, generation likely got cut off
    return "".join(buf).strip()

FALLBACK_ANSWER_RE = re.compile(r"final answer is[:\s]*\$?([^\n\$\.]+)\$?", re.IGNORECASE)

def extract_answer(generation: str) -> Optional[str]:
    boxed = extract_boxed(generation)
    if boxed is not None:
        return boxed
    m = FALLBACK_ANSWER_RE.search(generation)
    if m:
        return m.group(1).strip()
    return None

# Quick sanity check
assert extract_boxed(r"blah \boxed{\frac{1}{2}} blah") == r"\frac{1}{2}"
assert extract_boxed(r"The final answer is $\boxed{42}$.") == "42"
assert extract_boxed("no boxed answer here") is None
print("extract_boxed sanity checks passed.")



## 8. Math-equivalence verifier

This is the core "verifier" logic. Naive string equality massively under-counts correct
answers (`1/2` vs `\frac{1}{2}` vs `0.5`), so we normalize LaTeX and fall back to symbolic
comparison with `sympy` when direct string/numeric comparison doesn't resolve it.

The normalization + comparison strategy (strip `\left`/`\right`, `\!`, whitespace; convert
`\dfrac`/`\tfrac`→`\frac`; strip units and `\text{...}`; compare as sympy expressions; compare
as floats with tolerance; compare ordered tuples/intervals as strings) mirrors the approach
used by the original MATH paper's grader and by open evaluation harnesses.


In [ ]:

import sympy
from sympy.parsing.latex import parse_latex

def _normalize_latex(s: str) -> str:
    if s is None:
        return s
    s = s.strip()
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("\\!", "").replace("\\,", "").replace("\\ ", "")
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = re.sub(r"\\text\{.*?\}", "", s)
    s = re.sub(r"\\mbox\{.*?\}", "", s)
    s = s.replace("^{\\circ}", "").replace("^\\circ", "")
    s = s.replace("\\%", "").replace("%", "")
    s = s.replace("$", "")
    s = s.strip().strip(".")
    # Normalize simple a/b written without \frac
    s = re.sub(r"\s+", "", s)
    return s

def _to_float(s: str) -> Optional[float]:
    try:
        return float(s)
    except (TypeError, ValueError):
        return None

def _sympy_equal(a: str, b: str) -> bool:
    try:
        expr_a = parse_latex(a)
        expr_b = parse_latex(b)
        diff = sympy.simplify(expr_a - expr_b)
        return diff == 0
    except Exception:
        return False

def is_equiv(model_answer: Optional[str], gold_answer: Optional[str], tol: float = 1e-4) -> bool:
    # Return True if model_answer is mathematically equivalent to gold_answer.
    if model_answer is None or gold_answer is None:
        return False

    a_norm = _normalize_latex(model_answer)
    b_norm = _normalize_latex(gold_answer)

    if a_norm == b_norm:
        return True

    # Numeric comparison
    fa, fb = _to_float(a_norm), _to_float(b_norm)
    if fa is not None and fb is not None:
        return abs(fa - fb) < tol

    # Ordered-tuple / interval comparison, e.g. "(1,2)" vs "(1, 2)"
    strip_a = a_norm.replace("(", "").replace(")", "").replace("[", "").replace("]", "")
    strip_b = b_norm.replace("(", "").replace(")", "").replace("[", "").replace("]", "")
    if strip_a == strip_b and strip_a != a_norm:
        return True

    # Symbolic comparison via sympy as last resort
    if _sympy_equal(a_norm, b_norm):
        return True

    return False


In [ ]:

# Verifier self-test on representative MATH-style edge cases
test_cases = [
    ("\\frac{1}{2}", "0.5", True),
    ("1/2", "\\frac{1}{2}", True),
    ("42", "42.0", True),
    ("\\frac{3}{4}", "\\frac{6}{8}", True),
    ("x=3", "3", False),   # deliberately different formatting -> should NOT trivially pass
    ("(1,2)", "(1, 2)", True),
    ("\\sqrt{4}", "2", True),
    ("13", "31", False),
]

for model_ans, gold_ans, expected in test_cases:
    result = is_equiv(model_ans, gold_ans)
    status = "OK" if result == expected else "MISMATCH"
    print(f"[{status}] is_equiv({model_ans!r}, {gold_ans!r}) = {result} (expected {expected})")


## 9. Run the evaluation loop

In [ ]:

def run_evaluation(dataset, cfg: EvalConfig) -> pd.DataFrame:
    # Iterates over the dataset in batches, generates, extracts, and verifies each answer.
    records = []
    indices = list(range(len(dataset)))

    for start in tqdm(range(0, len(indices), cfg.batch_size), desc="Evaluating"):
        batch_idx = indices[start:start + cfg.batch_size]
        batch = dataset.select(batch_idx)
        prompts = [build_prompt(p, cfg.num_few_shot) for p in batch["problem"]]

        try:
            generations = generate_batch(prompts, cfg)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            # Retry one-by-one on OOM so a single bad batch doesn't kill the whole run
            generations = []
            for p in prompts:
                generations.extend(generate_batch([p], cfg))

        for i, gen in zip(batch_idx, generations):
            ex = dataset[i]
            gold = ex.get("answer")
            pred = extract_answer(gen)
            correct = is_equiv(pred, gold)
            records.append({
                "index": i,
                "problem": ex["problem"],
                "gold_answer": gold,
                "model_answer": pred,
                "correct": correct,
                "level": ex.get("level"),
                "subject": ex.get("subject") or ex.get("type"),
                "generation": gen,
            })

    return pd.DataFrame.from_records(records)

results_df = run_evaluation(eval_dataset, config)
results_df.head()


## 10. Aggregate metrics

In [ ]:

overall_acc = results_df["correct"].mean()
n_parsed = results_df["model_answer"].notna().mean()

print(f"Model: {config.model_name}")
print(f"Problems evaluated: {len(results_df)}")
print(f"Overall accuracy: {overall_acc:.2%}")
print(f"Answer-extraction rate (non-empty \\boxed{{}} found): {n_parsed:.2%}")


In [ ]:

acc_by_level = results_df.groupby("level")["correct"].agg(["mean", "count"]).rename(
    columns={"mean": "accuracy", "count": "n"}
).sort_index()
acc_by_level


In [ ]:

acc_by_subject = results_df.groupby("subject")["correct"].agg(["mean", "count"]).rename(
    columns={"mean": "accuracy", "count": "n"}
).sort_values("accuracy", ascending=False)
acc_by_subject


## 11. Visualizations

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

acc_by_level["accuracy"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title(f"{config.model_name}\nAccuracy by MATH difficulty level")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=0)

acc_by_subject["accuracy"].plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Accuracy by subject")
axes[1].set_xlabel("Accuracy")
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, "accuracy_breakdown.png"), dpi=150)
plt.show()


## 12. Error analysis

Inspect a sample of failures to sanity-check the *verifier itself*, not just the model.

In [ ]:

wrong = results_df[~results_df["correct"]]
print(f"{len(wrong)} / {len(results_df)} incorrect.")

for _, row in wrong.sample(min(5, len(wrong)), random_state=0).iterrows():
    print("=" * 100)
    print("PROBLEM:", textwrap.shorten(row["problem"], 300))
    print(f"GOLD: {row['gold_answer']!r}   |   MODEL PARSED: {row['model_answer']!r}")
    print("-" * 100)
    print("RAW GENERATION (truncated):")
    print(textwrap.shorten(row["generation"], 500))


## 13. Save results

In [ ]:

csv_path = os.path.join(config.output_dir, f"{config.model_name.replace('/', '_')}_math500_results.csv")
results_df.to_csv(csv_path, index=False)

summary = {
    "model": config.model_name,
    "dataset": config.dataset_name,
    "n_evaluated": len(results_df),
    "overall_accuracy": overall_acc,
    "answer_extraction_rate": n_parsed,
    "accuracy_by_level": acc_by_level["accuracy"].to_dict(),
    "accuracy_by_subject": acc_by_subject["accuracy"].to_dict(),
    "config": asdict(config),
}
summary_path = os.path.join(config.output_dir, "summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Saved per-example results to: {csv_path}")
print(f"Saved summary to: {summary_path}")



## 14. (Optional) Compare against other model sizes

Since `SmolLM-360M` is a base model, absolute accuracy will likely be low. To put the number
in context, re-run cells 4–13 after changing `config.model_name` to e.g.:

- `HuggingFaceTB/SmolLM2-360M-Instruct` (same size, instruction-tuned — usually a large jump)
- `HuggingFaceTB/SmolLM2-1.7B-Instruct` (bigger, instruction-tuned)

The helper below lets you run several models back-to-back and compare them without copy-pasting
the whole pipeline.


In [ ]:

def evaluate_model(model_name: str, cfg: EvalConfig, dataset) -> Dict[str, Any]:
    global tokenizer, model  # reuse the generation/verifier functions defined above
    local_cfg = EvalConfig(**{**asdict(cfg), "model_name": model_name})

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype_map[local_cfg.dtype]).to(DEVICE)
    model.eval()

    df = run_evaluation(dataset, local_cfg)
    acc = df["correct"].mean()
    del model
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return {"model": model_name, "accuracy": acc, "n": len(df), "df": df}

# Example (commented out — uncomment to actually run; each model download can be large):
# comparison = [
#     evaluate_model("HuggingFaceTB/SmolLM-360M", config, eval_dataset),
#     evaluate_model("HuggingFaceTB/SmolLM2-360M-Instruct", config, eval_dataset),
# ]
# pd.DataFrame([{"model": c["model"], "accuracy": c["accuracy"], "n": c["n"]} for c in comparison])



## Notes & caveats

- **Greedy decoding (`temperature=0.0`)** is used by default for reproducibility; MATH500
  leaderboards typically report greedy pass@1.
- The **few-shot prompt** matters a lot for base models — feel free to add/remove exemplars in
  `FEW_SHOT_EXEMPLARS` or tune `num_few_shot`.
- The **verifier** intentionally rejects things like `x=3` vs `3` as non-equivalent by default;
  if your model tends to answer in a different-but-valid form, inspect `results_df[~results_df.correct]`
  and extend `is_equiv` / `_normalize_latex` rather than loosening equality globally (a looser
  checker can silently inflate accuracy).
- Start with `config.num_samples = 20–50` to validate the pipeline runs end-to-end quickly
  before committing to the full 500-problem run, which can take a while on CPU.
